---

Reawakening

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# import pgl commands
from pgl import pgl, pglExperiment, pglEyeTrackingCalibrationTask, pglMessageAckTask, pglImageDatabase, pglMessages, pglTask, pglLabJack, pglParameter

# import other libraries
import numpy as np

# initialize pgl
pgl = pgl()
pgl.cleanUp()

================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.14 | packaged by conda-forge | (main, Aug 21 2026, 22:39:36) [Clang 19.1.7 ]
(pgl:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.6.2
(pgl:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl:checkOS)   Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pglBase) Main library instance created
(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Conta

---

Display settings

---

In [ ]:
# display settings - LCD should show up as SAMSUNG (select from top)
pgl.displaySettings()

---

Settings

---

In [ ]:
# Experiment settings, select reawakening
pgl.settings()

---

Eye tracker settings

---

In [ ]:
# set eye tracker settings, for example how many calibration points and the screen area to use for calibration
pgl.eyeTrackerSettings(settingsName='reawakening')

---

Search task

---

In [ ]:
class pglSearchTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Search Task"
        self.settings.nTrials = 100
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.config.imagesDirectory = "/Users/justin/Desktop/reawakening/stimulus/chair/0001"
        self.settings.config.imageHeight= 15
        
        # setup digitalIO
        #from pgl import pglLabJack
        #self.digIO = pglLabJack()
        
        # set seglens, 
        # 1st segment is fixation
        # 2nd segment is image display
        # 2nd segment is ISI
        self.settings.seglen = [0.5, 3.0, 0.5]

        # initialize image database using parameters from config
        self.state._imdb = pglImageDatabase(self.settings.config.imagesDirectory)
        if self.state._imdb.nStimuli==0:
            pglMessages.warning(f"No images found in {self.state._imdb.dataPath}")
            return
                
        # preload images
        self.state._imdb.preload()            

        # add parameter for image number
        imageNumParameter = pglParameter('imageNum',np.arange(self.state._imdb.nStimuli))
        self.addParameter(imageNumParameter)
        
        imageNumParameter.print()
    
    ########################
    def configure(self,e):
        '''
        configure the task
        '''
        task = e.getLastRun(task=self)
        print("*************************")
        if task is not None: task.print()
        print("*************************")
        
    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment == 0: 
            # set fixation color to white
            self.state.fixColor = 1
            # get the current image number
            imageNum = self.currentParams['imageNum']
            # get the image data
            img = self.state._imdb.get(imageNum)
            img.convert("RGB")
            print(f"img: {img}")
            # turn into a pglImage
            self.state._currentImage = self.pgl.imageCreate(np.array(img))

    ########################
    # handleSubjectResponse
    ########################    
    def handleSubjectResponse(self, response, updateTime):
        '''
        Handle the subject response. Response will come in as an integer
        value of what button was pressed. The order of buttons is set
        in pgl.settings() in the field "responseKeys"
        '''
        # already received a response
        if self.state.gotResponse: return None
        # mark that we got a response
        self.state.gotResponse = True
        
        # return response type
        return True

    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment == 1: 
            if self.state._currentImage:
                self.state._currentImage.display(height=self.settings.config.imageHeight)
        else:
            pgl.fixationABC()


---

Setup experiment

---

In [30]:
# clean up any open windows
pgl.cleanUp()

#e = pglExperiment(pgl,experimentName='Search Task',settingsName='reawakening')
e = pglExperiment(pgl,experimentName='Search Task',settingsName='window')

# First run a calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)

# add the search task to the experiment
messageAckTask = pglMessageAckTask(pgl, "Press space to start search task")
e.addTask(messageAckTask,addPhase=True)
searchTask = pglSearchTask(pgl)
e.addTask(searchTask,addPhase=True)

# Run a final calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)


(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Containers/gru.mglMetal/Data
(pgl->pglSettingsManager:getSettings) Loading settings from '/Users/justin/.pgl/settings/window.json'.
(pglImageDatabase->pglStimulusDatabase:__init__) Found 100 stimulus files in /Users/justin/Desktop/reawakening/stimulus/chair/0001
(pglImageFile:_loadImage) Loading image: //Users/justin/Desktop/reawakening/stimulus/chair/0001/000.jpg
(pglImageFile:_loadImage) Loading image: //Users/justin/Desktop/reawakening/stimulus/chair/0001/001.jpg
(pglImageFile:_loadImage) Loading image: //Users/justin/Desktop/reawakening/stimulus/chair/0001/002.jpg
(pglImageFile:_loadImage) Loading image: //Users/justin/Desktop/reawakening/stimulus/chair/0001/003.jpg
(pglImageFile:_loadImage) Loading image: //Users/justin/Desktop/reawakening/stimulus/chair/0001/004.jpg
(pglImageFile:_loadImage) Loading image: //Users/justin/Desktop/reawakening/stimulus/chair/0001/005.jpg
(pglImageFile:_loadImage) Load

---

Run task

---

In [ ]:
e.initScreen()
e.run()
e.display()


(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Containers/gru.mglMetal/Data
================================= pglBase:open =================================
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pglBase:getMetalAppName) Using latest build: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pgl->pglBase:open) Starting mglMetal application: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pgl->pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260916_233514.aADJDRy0dc
(pgl:_pglComm) .Connected to: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260916_233514.aADJDRy0dc
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pglKeyboardMouse:start) Starting keyboard and mouse event listener.
(pglEventListener) Eating 13 keys: ['1', '2', '3', '4', '5', '<keycode

---

Test Labjack

---

In [ ]:
from pgl import pgl, pglLabJack
pgl = pgl()
labJack = pglLabJack()

In [ ]:
labJack.setupDigitalOutput(channel=0, group="FI0", pulseLen=3)

In [ ]:
labJack.digitalOutputPulse(0)

Self-initatied trial
Do not require fixation
Cross on the gray screen - press space bar - initiates stimulus 35x35 if you think there is a target press j if you do not think there is the target press j. As soon as you press the key it goes to the gray screen with a fixation cross.\
Every 25th
Calibration task
5 minutes for 100 images (3 seconds per image)
10 minutes for 100 images (6 seconds per image)

Instead to fixed interval for fixation 0.5s then 2s per image (Eckstein timing) 1s 


In [27]:
r = e.getLastRun()

(pglRun:data) Loading data for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46_23-32-04
(pglRun:data) Loading data for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46
(pglRun:data) Loading data for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-26-38
(pglRun:data) Loading data for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_22-46-09
(pglRun:data) Loading data for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_22-15-42
(pglRun:data) Loading data for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46_23-32-40


In [28]:
r.print()

(pglRun:experimentSettings) Loading experiment settings for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46_23-32-40
Experiment: Search Task | Subject ID: s0000
Duration: 799ms
Number of volume triggers: 0
taskName: acknowledgeMessage
taskName: eyeTrackingCalibration
taskName: acknowledgeMessage
taskName: searchTask
taskName: acknowledgeMessage
taskName: eyeTrackingCalibration
GOT NONE HERE
(pglRun:tasks) Loading tasks for: //Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46_23-32-40
taskName: acknowledgeMessage
(pglTask:load) Loading task data from: /Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46_23-32-40/acknowledgeMessage
taskName: eyeTrackingCalibration
(pglTask:load) Loading task data from: /Users/justin/data/searchTask/s0000/session_2026-09-16/run_23-28-46_23-32-40/eyeTrackingCalibration
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglBase:validateFilesystem) /Users/justin/data/searchT

AttributeError: 'NoneType' object has no attribute 'print'